# Lab 3 — YOLOv8s trên Kaggle (2 × T4)

Notebook độc lập: chứa toàn bộ implementation, Dataset adapter và evaluator.
**Không cần upload thư mục code hoặc notebook còn lại.**

1. Tạo Kaggle Notebook, chọn **GPU T4 ×2**, bật **Internet**.
2. Attach Kaggle Dataset đã tạo từ `varroa_kaggle_ready.zip`.
3. Import notebook này rồi **Run All**. Đường dẫn dataset được tìm tự động.
4. Tải ZIP kết quả trong `/kaggle/working/` sau khi hoàn tất.

Notebook tự chạy thử một batch train và 4 ảnh validation trên đúng GPU/AMP/worker
đã chọn trước khi train đầy đủ. Lượt thử dùng output riêng, không đọc test và không
thay đổi weights khởi tạo của run chính. Nếu thử thất bại, notebook dừng tại lỗi đó.

Model: COCO-pretrained, fine-tune một class `varroa`; **seed 42**, tối đa 100 epochs,
early stopping 15 epochs không cải thiện COCO mAP@50:95. Global batch 8 = 4 ảnh/GPU.
Hai notebook dùng cùng code preprocessing: RGB, letterbox 640 × 640 giữ tỷ lệ,
padding 114, lật ngang p=0.5 và brightness/contrast/color trong [0.9,1.1].
Không mosaic, crop hay multi-scale. Validation/test không augmentation.

Chọn checkpoint theo **pycocotools COCO AP@[.50:.05:.95]** trên validation;
chọn confidence theo F1 validation tại IoU=.5 rồi khóa khi chạy test.
Tốc độ đo riêng trên **một T4, FP32, batch 1**, gồm preprocess, chuyển GPU,
model, NMS và chuyển box về ảnh gốc; không gồm đọc file/giải mã ảnh.

YOLO dùng EMA của Ultralytics; Faster R-CNN dùng weights trực tiếp. Đây là khác biệt recipe được ghi lại, không phải phép cô lập kiến trúc tuyệt đối.

Internet cần cho cài thư viện và tải pretrained weights lần đầu. Resume giữ trạng thái
optimizer/scheduler/scaler theo framework, không cam kết giống bit khi worker khởi động lại.
Seed và cuDNN được cố định; không ép mọi kernel deterministic để giữ CUDA ROIAlign
chuẩn của Torchvision. Một số phép toán detection trên GPU vẫn có sai khác nhỏ.
Một seed không cho phép kết luận về độ biến thiên giữa các lần huấn luyện.


In [ ]:
# 1. Cấu hình — chỉ đổi nếu bạn muốn thay ngân sách training.
from pathlib import Path
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["WANDB_MODE"] = "disabled"
os.environ["COMET_MODE"] = "DISABLED"
os.environ["YOLO_CONFIG_DIR"] = "/kaggle/working/.ultralytics"

DATASET_ROOT = None  # Tự tìm dưới /kaggle/input; hoặc đặt đường dẫn dataset root.
EPOCHS = 100
PATIENCE = 15
GLOBAL_BATCH = 8
GPUS = 2
WORKERS = 2  # Mỗi GPU; giảm về 0 nếu dataloader gặp lỗi shared memory.
EVAL_BATCH = 4
SEED = 42
IMAGE_SIZE = 640
AMP = True
RESUME_CHECKPOINT = None  # Đường dẫn last.pt của một run CHƯA train xong.
OUTPUT = Path("/kaggle/working/yolov8s_seed42")
RUNTIME = Path("/kaggle/working/lab3_yolov8s_runtime")
RUNTIME.mkdir(parents=True,exist_ok=True)
OUTPUT.mkdir(parents=True,exist_ok=True)
if not RESUME_CHECKPOINT and any((OUTPUT/name).exists() for name in ("best.pt","last.pt","weights/best.pt","weights/last.pt")):
    raise RuntimeError("Output đã có checkpoint. Chọn OUTPUT mới hoặc đặt RESUME_CHECKPOINT=last.pt.")


In [ ]:
# 2. Cài các dependency bổ sung; giữ cặp torch/torchvision sẵn có của Kaggle.
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","--quiet",'pycocotools==2.0.11','ultralytics==8.3.228'],check=True)
import torch, torchvision
assert torch.cuda.is_available(), "Chọn GPU T4 x2 trong Kaggle Settings."
assert torch.cuda.device_count() >= GPUS, "Notebook yêu cầu đủ 2 GPU; kiểm tra accelerator."
assert GLOBAL_BATCH % GPUS == 0
print("torch:",torch.__version__,"torchvision:",torchvision.__version__)
for index in range(GPUS):
    print(index,torch.cuda.get_device_name(index))


In [ ]:
# 3. Tìm dataset và ghi cấu hình vào vùng có quyền ghi.
import json, shutil, zipfile, hashlib, importlib.metadata

def find_dataset():
    if DATASET_ROOT:
        root=Path(DATASET_ROOT)
        assert (root/"annotations/instances_train.json").exists()
        return root.resolve()
    candidates=sorted({p.parent.parent for p in Path("/kaggle/input").rglob("instances_train.json")
                       if p.parent.name=="annotations" and (p.parent.parent/"data.yaml").exists()})
    if not candidates:
        archives=list(Path("/kaggle/input").rglob("varroa_kaggle_ready.zip"))
        assert len(archives)==1, "Attach varroa_kaggle_ready dataset hoặc đặt DATASET_ROOT."
        root=Path("/kaggle/working/varroa_data")
        root.mkdir(exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            for member in archive.namelist():
                assert (root/member).resolve().is_relative_to(root.resolve()), "Unsafe ZIP path"
            archive.extractall(root)
        candidates=[root]
    assert len(candidates)==1, f"Tìm thấy {len(candidates)} dataset; đặt DATASET_ROOT để chọn."
    return candidates[0].resolve()

root=find_dataset()
for split in ("train","val","test"):
    data=json.loads((root/"annotations"/f"instances_{split}.json").read_text())
    assert data["categories"]==[{"id":1,"name":"varroa","supercategory":"mite"}]
    assert all((root/im["file_name"]).is_file() for im in data["images"])
    print(split,len(data["images"]),"images;",len(data["annotations"]),"boxes")
runtime_yaml=RUNTIME/"data.yaml"
runtime_yaml.write_text(f'path: {json.dumps(str(root))}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: 1\nnames: [varroa]\n')


In [ ]:
CONFIG = dict(root=str(root),data_yaml=str(runtime_yaml),output=str(OUTPUT),
    epochs=EPOCHS,patience=PATIENCE,batch=GLOBAL_BATCH,gpus=GPUS,workers=WORKERS,
    seed=SEED,size=IMAGE_SIZE,amp=AMP,learning_rate=0.01,warmup_epochs=3,
    eval_batch=EVAL_BATCH,eval_device="cuda:0",benchmark_samples=100,
    resume=RESUME_CHECKPOINT,weights=str(RUNTIME/"yolov8s.pt"))
CONFIG_PATH=RUNTIME/"config.json"
CONFIG_PATH.write_text(json.dumps(CONFIG,indent=2))
(OUTPUT/"config.json").write_text(json.dumps(CONFIG,indent=2))
environment=dict(python=sys.version,torch=torch.__version__,torchvision=torchvision.__version__,
    CUDA=torch.version.cuda,GPUs=[torch.cuda.get_device_name(i) for i in range(GPUS)],
    packages={p:importlib.metadata.version(p) for p in ['pycocotools', 'ultralytics']},
    annotation_sha256={s:hashlib.sha256((root/"annotations"/f"instances_{s}.json").read_bytes()).hexdigest()
                       for s in ("train","val","test")},
    evaluator="pycocotools",score_floor=0.001,NMS_IoU=0.5,max_detections=100,
    precision_eval="FP32",EMA=True,native_detection_kernels=True,
    normalization='RGB / 255')
(OUTPUT/"environment.json").write_text(json.dumps(environment,indent=2))
(OUTPUT/"requirements_freeze.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))
print("Dataset:",root,"\nOutput:",OUTPUT)


## Implementation dùng chung

Cell tiếp theo ghi module vào `/kaggle/working/`. Đó là mã được đóng gói ngay trong
notebook, không cần tìm file trên máy local. YOLO và Faster R-CNN dùng **cùng module này**.
COCO AP giữ score floor .001; P/R/F1 dùng ngưỡng được chọn riêng trên validation.
Mean IoU chỉ tính trên TP ghép một-một ở IoU≥.5; đọc cùng Recall.
AP/AR của nhóm không có ground truth được xuất `null`, không coi là điểm 0.


In [ ]:
%%writefile /kaggle/working/lab3_yolov8s_runtime/common.py
"""Shared preprocessing and evaluation for both standalone Kaggle notebooks."""
import contextlib
import csv
import io
import json
import os
import random
import time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageEnhance
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import Dataset

SCORE_FLOOR = 0.001
NMS_IOU = 0.5
MAX_DETECTIONS = 100
PRIMARY_METRICS = ("mAP50_95", "AP50", "precision", "recall", "F1", "mean_IoU_TP", "FPS")


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n")


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # Forcing all deterministic ops makes Torchvision switch CUDA ROIAlign to a
    # different compiled implementation. Keep native detection kernels for both
    # the T4 training budget and the inference benchmark; seeds/cuDNN stay fixed.
    torch.use_deterministic_algorithms(False)
    cv2.setNumThreads(0)


def seed_worker(_):
    value = torch.initial_seed() % 2**32
    np.random.seed(value)
    random.seed(value)
    cv2.setNumThreads(0)


def letterbox(image, size=640):
    """RGB uint8 → RGB float CHW; return the actual x/y resize factors and padding."""
    width, height = image.size
    ratio = min(size / width, size / height)
    nw, nh = round(width * ratio), round(height * ratio)
    resized = cv2.resize(np.asarray(image), (nw, nh), interpolation=cv2.INTER_LINEAR)
    left, top = (size - nw) // 2, (size - nh) // 2
    canvas = np.full((size, size, 3), 114, dtype=np.uint8)
    canvas[top:top+nh, left:left+nw] = resized
    tensor = torch.from_numpy(canvas.transpose(2, 0, 1).copy()).float() / 255.0
    return tensor, (nw / width, nh / height, left, top, width, height)


def transform_boxes(boxes, geometry, inverse=False):
    boxes = boxes.clone().reshape(-1, 4)
    sx, sy, left, top, width, height = geometry
    if inverse:
        boxes[:, [0, 2]] = ((boxes[:, [0, 2]] - left) / sx).clamp(0, width)
        boxes[:, [1, 3]] = ((boxes[:, [1, 3]] - top) / sy).clamp(0, height)
    else:
        boxes[:, [0, 2]] = boxes[:, [0, 2]] * sx + left
        boxes[:, [1, 3]] = boxes[:, [1, 3]] * sy + top
    return boxes


class CocoDataset(Dataset):
    def __init__(self, root, split, size=640, augment=False):
        self.root, self.size, self.augment = Path(root), size, augment
        self.annotation_path = self.root / "annotations" / f"instances_{split}.json"
        self.data = json.loads(self.annotation_path.read_text())
        self.images = sorted(self.data["images"], key=lambda x: x["id"])
        self.annotations = defaultdict(list)
        for ann in self.data["annotations"]:
            assert ann["category_id"] == 1 and ann["iscrowd"] == 0
            self.annotations[ann["image_id"]].append(ann)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        entry = self.images[index]
        with Image.open(self.root / entry["file_name"]) as source:
            image = source.convert("RGB")
        boxes = torch.tensor([a["bbox"] for a in self.annotations[entry["id"]]], dtype=torch.float32).reshape(-1, 4)
        boxes[:, 2:] += boxes[:, :2]
        if self.augment:
            if random.random() < 0.5:
                image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
                x1, x2 = boxes[:, 0].clone(), boxes[:, 2].clone()
                boxes[:, 0], boxes[:, 2] = image.width - x2, image.width - x1
            for enhancer in (ImageEnhance.Brightness, ImageEnhance.Contrast, ImageEnhance.Color):
                image = enhancer(image).enhance(random.uniform(0.9, 1.1))
        tensor, geometry = letterbox(image, self.size)
        boxes = transform_boxes(boxes, geometry)
        target = {"boxes": boxes, "labels": torch.ones(len(boxes), dtype=torch.int64),
                  "image_id": torch.tensor(entry["id"]),
                  "area": (boxes[:, 2]-boxes[:, 0]) * (boxes[:, 3]-boxes[:, 1]),
                  "iscrowd": torch.zeros(len(boxes), dtype=torch.int64)}
        return tensor, target, geometry


def detection_collate(batch):
    images, targets, geometry = zip(*batch)
    return list(images), list(targets), list(geometry)


def coco_metrics(annotation_path, predictions):
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(str(annotation_path))
        if predictions:
            result = gt.loadRes(predictions)
        else:
            result = COCO()
            result.dataset = {"images": gt.dataset["images"], "categories": gt.dataset["categories"], "annotations": []}
            result.createIndex()
        evaluator = COCOeval(gt, result, "bbox")
        evaluator.params.maxDets = [1, 10, MAX_DETECTIONS]
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    names = ("mAP50_95", "AP50", "AP75", "AP_small", "AP_medium", "AP_large",
             "AR1", "AR10", "AR100", "AR_small", "AR_medium", "AR_large")
    return {name: (float(v) if v >= 0 else None) for name, v in zip(names, evaluator.stats)}


class Predictor:
    """Shared FP32 inference boundary: letterboxed tensors → original-image COCO boxes."""
    def __init__(self, model, family, device):
        self.model, self.family, self.device = model.float().eval(), family, torch.device(device)

    @torch.inference_mode()
    def __call__(self, images, geometry, ids):
        tensors = [im.to(self.device, non_blocking=True) for im in images]
        if self.family == "fasterrcnn":
            outputs = self.model(tensors)
            detections = [(o["boxes"], o["scores"]) for o in outputs]
        else:
            from ultralytics.utils.nms import non_max_suppression
            raw = self.model(torch.stack(tensors))
            raw = raw[0] if isinstance(raw, tuple) else raw
            outputs = non_max_suppression(raw, conf_thres=SCORE_FLOOR, iou_thres=NMS_IOU,
                                           max_det=MAX_DETECTIONS, nc=1)
            detections = [(o[:, :4], o[:, 4]) for o in outputs]
        records = []
        for (boxes, scores), geo, image_id in zip(detections, geometry, ids):
            boxes = transform_boxes(boxes, geo, inverse=True).cpu()
            scores = scores.detach().cpu()
            for box, score in zip(boxes.tolist(), scores.tolist()):
                x1, y1, x2, y2 = box
                if x2 > x1 and y2 > y1 and score >= SCORE_FLOOR:
                    records.append({"image_id": int(image_id), "category_id": 1,
                                    "bbox": [x1, y1, x2-x1, y2-y1], "score": score})
        return records


def predict_dataset(predictor, dataset, batch_size=8, workers=2):
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False,
        num_workers=workers, pin_memory=predictor.device.type == "cuda",
        multiprocessing_context="spawn" if workers else None,
        collate_fn=detection_collate, worker_init_fn=seed_worker)
    records = []
    for images, targets, geometry in loader:
        records.extend(predictor(images, geometry, [int(t["image_id"]) for t in targets]))
    return records


def matched_detections(annotation_path, predictions):
    """Greedy score-ordered, class-aware one-to-one matching at IoU=0.5."""
    data = json.loads(Path(annotation_path).read_text())
    truth = defaultdict(list)
    for ann in data["annotations"]:
        truth[ann["image_id"]].append(ann["bbox"])
    grouped = defaultdict(list)
    for p in predictions:
        grouped[p["image_id"]].append(p)
    records = []
    for image_id, preds in grouped.items():
        boxes = np.asarray(truth[image_id], dtype=float).reshape(-1, 4)
        used = np.zeros(len(boxes), dtype=bool)
        for p in sorted(preds, key=lambda x: -x["score"]):
            x,y,w,h = p["bbox"]
            if len(boxes):
                inter = np.maximum(0, np.minimum(x+w, boxes[:,0]+boxes[:,2])-np.maximum(x,boxes[:,0])) * np.maximum(
                    0, np.minimum(y+h, boxes[:,1]+boxes[:,3])-np.maximum(y,boxes[:,1]))
                ious = inter / np.maximum(w*h+boxes[:,2]*boxes[:,3]-inter, 1e-12)
                ious[used] = -1
                idx = int(ious.argmax())
                tp = bool(ious[idx] >= 0.5)
                matched_iou = float(ious[idx]) if tp else 0.0
                if tp:
                    used[idx] = True
            else:
                tp, matched_iou = False, 0.0
            records.append({"score": p["score"], "tp": int(tp), "iou": matched_iou, "image_id": image_id})
    return records, len(data["annotations"]), [im["id"] for im in data["images"] if not truth[im["id"]]], len(data["images"])


def select_threshold(annotation_path, predictions):
    records, total, _, _ = matched_detections(annotation_path, predictions)
    if not records:
        return 1.0, []
    records.sort(key=lambda r: -r["score"])
    tp, curve = 0, []
    for i, row in enumerate(records):
        tp += row["tp"]
        if i+1 < len(records) and records[i+1]["score"] == row["score"]:
            continue
        precision, recall = tp/(i+1), tp/max(total,1)
        f1 = 2*precision*recall/max(precision+recall,1e-12)
        curve.append({"threshold": row["score"], "precision": precision, "recall": recall, "f1": f1})
    best = max(curve, key=lambda r: (r["f1"], r["threshold"]))
    if best["f1"] == 0:
        return 1.0, curve
    return best["threshold"], curve


def operating_metrics(annotation_path, predictions, threshold):
    records, total, negatives, count = matched_detections(annotation_path, predictions)
    selected = [r for r in records if r["score"] >= threshold]
    tp = sum(r["tp"] for r in selected)
    fp, fn = len(selected)-tp, total-tp
    precision, recall = tp/max(tp+fp,1), tp/max(total,1)
    false_images = {r["image_id"] for r in selected} & set(negatives)
    return {"confidence_threshold": threshold, "matching_IoU": 0.5,
            "precision": precision, "recall": recall,
            "F1": 2*precision*recall/max(precision+recall,1e-12),
            "TP": tp, "FP": fp, "FN": fn,
            "mean_IoU_TP": sum(r["iou"] for r in selected)/tp if tp else None,
            "negative_image_false_alarm_rate": len(false_images)/len(negatives) if negatives else None,
            "false_positives_per_image": fp/count}


def benchmark(predictor, dataset, samples=100, warmup=20):
    """Batch 1, FP32, one GPU; excludes file decoding, includes all other inference steps."""
    indices = np.linspace(0, len(dataset)-1, min(samples, len(dataset)), dtype=int)
    originals = []
    for index in indices:
        entry = dataset.images[index]
        with Image.open(dataset.root / entry["file_name"]) as im:
            originals.append((im.convert("RGB"), entry["id"]))
    cuda = predictor.device.type == "cuda"
    def one(item):
        tensor, geo = letterbox(item[0], dataset.size)
        return predictor([tensor], [geo], [item[1]])
    for i in range(warmup):
        one(originals[i % len(originals)])
    if cuda:
        torch.cuda.synchronize(predictor.device)
        torch.cuda.reset_peak_memory_stats(predictor.device)
    elapsed = []
    for item in originals:
        if cuda:
            torch.cuda.synchronize(predictor.device)
        start = time.perf_counter()
        one(item)
        if cuda:
            torch.cuda.synchronize(predictor.device)
        elapsed.append((time.perf_counter()-start)*1000)
    return {"latency_mean_ms": float(np.mean(elapsed)), "latency_median_ms": float(np.median(elapsed)),
            "latency_p95_ms": float(np.percentile(elapsed,95)), "FPS": 1000/float(np.mean(elapsed)),
            "inference_peak_VRAM_MB": torch.cuda.max_memory_allocated(predictor.device)/1024**2 if cuda else None,
            "latency_samples": len(elapsed), "batch_size": 1, "inference_precision": "FP32",
            "timing_scope": "predecoded RGB -> letterbox -> H2D -> model -> NMS -> original coords -> CPU records; excludes disk/decode"}


def render_examples(dataset, predictions, threshold, path):
    grouped = defaultdict(list)
    for p in predictions:
        if p["score"] >= threshold:
            grouped[p["image_id"]].append(p)
    positive = [i for i in dataset.images if dataset.annotations[i["id"]]][:8]
    negative = [i for i in dataset.images if not dataset.annotations[i["id"]]][:8]
    sheet = Image.new("RGB", (640, 4*310), "white")
    for k, entry in enumerate(positive+negative):
        with Image.open(dataset.root/entry["file_name"]) as source:
            im = source.convert("RGB")
        draw = ImageDraw.Draw(im)
        for ann in dataset.annotations[entry["id"]]:
            x,y,w,h = ann["bbox"]
            draw.rectangle((x,y,x+w,y+h), outline="lime", width=2)
        for p in grouped[entry["id"]]:
            x,y,w,h = p["bbox"]
            draw.rectangle((x,y,x+w,y+h), outline="red", width=1)
        sheet.paste(im, ((k%4)*160,(k//4)*310+20))
        ImageDraw.Draw(sheet).text(((k%4)*160,(k//4)*310+3),f'id={entry["id"]}',fill="black")
    sheet.save(path)


def final_evaluation(model, family, config):
    out = Path(config["output"])
    predictor = Predictor(model.to(config["eval_device"]), family, config["eval_device"])
    val = CocoDataset(config["root"], "val", config["size"])
    test = CocoDataset(config["root"], "test", config["size"])
    val_predictions = predict_dataset(predictor, val, config["eval_batch"], config["workers"])
    threshold, curve = select_threshold(val.annotation_path, val_predictions)
    save_json(out/"validation_predictions.json", val_predictions)
    save_json(out/"confidence_selection.json", {"selected_on": "validation", "rule": "maximum F1 at IoU=0.5; ties prefer higher confidence", "threshold": threshold})
    save_json(out/"validation_PR_F1_curve.json", curve)
    # The threshold is locked before reading test predictions.
    predictions = predict_dataset(predictor, test, config["eval_batch"], config["workers"])
    save_json(out/"test_predictions.json", predictions)
    metrics = {"model": family, "seed": config["seed"],
               "validation": {**coco_metrics(val.annotation_path,val_predictions), **operating_metrics(val.annotation_path,val_predictions,threshold)},
               "test": {**coco_metrics(test.annotation_path,predictions), **operating_metrics(test.annotation_path,predictions,threshold)},
               "parameters": sum(p.numel() for p in model.parameters()),
               "benchmark": benchmark(predictor,test,config["benchmark_samples"])}
    save_json(out/"metrics.json",metrics)
    row = {"model":family,"seed":config["seed"],
           **{key:metrics["test"][key] for key in PRIMARY_METRICS if key != "FPS"},
           "FPS":metrics["benchmark"]["FPS"]}
    with (out/"comparison.csv").open("w",newline="") as f:
        writer=csv.DictWriter(f,fieldnames=list(row));writer.writeheader();writer.writerow(row)
    render_examples(test,predictions,threshold,out/"test_examples.jpg")
    print(json.dumps(metrics,indent=2))
    return metrics


def prepare_gpu_smoke_config(config, runtime):
    """Prepare a tiny train/val subset for the actual two-GPU/AMP/worker preflight.

    Never samples test or changes the real training dataset/config/output.
    """
    runtime = Path(runtime)
    source = Path(config["root"])
    root = runtime / "gpu_smoke_data"
    (root / "annotations").mkdir(parents=True, exist_ok=True)
    image_link = root / "images"
    target = (source / "images").resolve()
    if image_link.is_symlink() and image_link.resolve() != target:
        image_link.unlink()
    if not image_link.is_symlink():
        image_link.symlink_to(target, target_is_directory=True)
    for split, count in (("train", max(config["batch"], 8)), ("val", 4)):
        data = json.loads((source / "annotations" / f"instances_{split}.json").read_text())
        positives = {a["image_id"] for a in data["annotations"]}
        images = sorted(data["images"], key=lambda im:im["id"])
        take = [im for im in images if im["id"] in positives][:count//2]
        take += [im for im in images if im["id"] not in positives][:count-count//2]
        selected_ids = {im["id"] for im in take}
        if len(take) != count:
            raise ValueError(f"Not enough positive/negative images for {split} smoke test")
        data["images"] = take
        data["annotations"] = [a for a in data["annotations"] if a["image_id"] in selected_ids]
        save_json(root / "annotations" / f"instances_{split}.json", data)
    yaml = root / "data.yaml"
    yaml.write_text(f'path: {json.dumps(str(root.resolve()))}\ntrain: images/train\nval: images/val\nnames: [varroa]\nnc: 1\n')
    smoke = {**config, "root":str(root.resolve()), "data_yaml":str(yaml.resolve()),
             "output":str((runtime / "gpu_smoke_run").resolve()),
             "epochs":1, "patience":1, "warmup_epochs":0, "resume":None}
    path = runtime / "gpu_smoke_config.json"
    save_json(path, smoke)
    return path


## Implementation YOLOv8s

Subclass trainer Ultralytics dùng COCO Dataset chung, loss/head YOLOv8s gốc, và COCO validation chung làm fitness để lưu best.pt. Khởi chạy bằng torchrun để mỗi rank dùng đúng subclass.

In [ ]:
%%writefile /kaggle/working/lab3_yolov8s_runtime/yolov8_train.py
"""YOLOv8s with the same COCO Dataset adapter and evaluator as Faster R-CNN.

Pin ultralytics==8.3.228: the trainer extension is tested against that implementation.
Launch explicitly with torchrun so the custom subclass is imported on every rank.
"""
import argparse
import json
import os
import time
from pathlib import Path

import torch
import torch.distributed as dist
from torch.utils.data import DataLoader, DistributedSampler
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

from common import (CocoDataset, Predictor, coco_metrics, final_evaluation,
                    predict_dataset, save_json, seed_everything, seed_worker)


class YOLOCocoDataset(CocoDataset):
    """Keep the input pipeline identical; adapt targets only at the model boundary."""
    @staticmethod
    def collate_fn(batch):
        images,targets,geometry=zip(*batch)
        bboxes,classes,indices=[],[],[]
        for i,(image,target) in enumerate(zip(images,targets)):
            xyxy=target["boxes"]
            xywh=torch.cat(((xyxy[:,:2]+xyxy[:,2:])/2,xyxy[:,2:]-xyxy[:,:2]),dim=1)/image.shape[-1]
            bboxes.append(xywh)
            classes.append(torch.zeros((len(xywh),1),dtype=torch.float32))
            indices.append(torch.full((len(xywh),),i,dtype=torch.float32))
        return {"img":torch.stack(images),"bboxes":torch.cat(bboxes),
                "cls":torch.cat(classes),"batch_idx":torch.cat(indices),
                "im_file":[str(int(t["image_id"])) for t in targets],
                "ori_shape":[(g[5],g[4]) for g in geometry],
                "resized_shape":[tuple(image.shape[-2:]) for image in images]}


class LabTrainer(DetectionTrainer):
    def __init__(self,*args,**kwargs):
        self.lab_config=json.loads(Path(os.environ["LAB_CONFIG"]).read_text())
        super().__init__(*args,**kwargs)

    def get_dataloader(self,dataset_path,batch_size=16,rank=0,mode="train"):
        c=self.lab_config
        dataset=YOLOCocoDataset(c["root"],mode,c["size"],augment=mode=="train")
        world=int(os.environ.get("WORLD_SIZE","1"))
        global_rank=int(os.environ.get("RANK","0"))
        sampler=DistributedSampler(dataset,num_replicas=world,rank=global_rank,
            shuffle=mode=="train",seed=c["seed"]) if world>1 else None
        return DataLoader(dataset,batch_size=batch_size,shuffle=mode=="train" and sampler is None,
            sampler=sampler,num_workers=c["workers"],pin_memory=True,
            multiprocessing_context="spawn" if c["workers"] else None,
            collate_fn=YOLOCocoDataset.collate_fn,worker_init_fn=seed_worker,
            generator=torch.Generator().manual_seed(c["seed"]+global_rank))

    def check_resume(self,overrides):
        super().check_resume(overrides)
        if self.resume:
            c=self.lab_config
            for key,expected in (("seed",c["seed"]),("imgsz",c["size"]),
                                 ("batch",c["batch"]),("epochs",c["epochs"])):
                if getattr(self.args,key)!=expected:
                    raise ValueError(f"Resume requires unchanged {key}")
            # Native resume otherwise retains paths stored in the old Kaggle session.
            self.args.data=c["data_yaml"]
            self.args.project=str(Path(c["output"]).parent)
            self.args.name=Path(c["output"]).name
            self.args.save_dir=c["output"]
            if not (Path(c["output"])/"weights/best.pt").exists():
                raise ValueError("Copy the previous weights/best.pt into the new output before resuming.")

    def preprocess_batch(self,batch):
        # Shared Dataset already supplies float RGB in [0,1], do not divide again.
        return {k:v.to(self.device,non_blocking=True) if isinstance(v,torch.Tensor) else v
                for k,v in batch.items()}

    def validate(self):
        c=self.lab_config
        rank=int(os.environ.get("RANK","0"))
        world=int(os.environ.get("WORLD_SIZE","1"))
        ap=torch.zeros(3,device=self.device)
        if rank==0:
            dataset=CocoDataset(c["root"],"val",c["size"])
            model=self.ema.ema if self.ema else self.model
            predictions=predict_dataset(Predictor(model,"yolov8s",self.device),dataset,
                c["eval_batch"],c["workers"])
            metrics=coco_metrics(dataset.annotation_path,predictions)
            ap[:]=torch.tensor([metrics["mAP50_95"],metrics["AP50"],metrics["AP75"]],device=self.device)
            save_json(Path(c["output"])/"last_validation_metrics.json",metrics)
        if world>1:
            dist.broadcast(ap,src=0)
        fitness=float(ap[0])
        if self.best_fitness is None or fitness>self.best_fitness:
            self.best_fitness=fitness
        return {"val/COCO_mAP50_95":fitness,"val/COCO_AP50":float(ap[1]),
                "val/COCO_AP75":float(ap[2])},fitness

    def final_eval(self):
        # Preserve full last.pt for resume. A separate one-GPU process evaluates best.pt.
        # Native final_eval would invoke a different evaluator and strip optimizer state.
        pass


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument("--config",required=True)
    parser.add_argument("--evaluate",action="store_true")
    args=parser.parse_args()
    c=json.loads(Path(args.config).read_text())
    os.environ["LAB_CONFIG"]=str(Path(args.config).resolve())
    if args.evaluate:
        seed_everything(c["seed"])
        model=YOLO(str(Path(c["output"])/"weights/best.pt")).model
        final_evaluation(model,"yolov8s",c)
        return
    if not torch.cuda.is_available():
        raise RuntimeError("Enable GPU T4 x2 in Kaggle settings.")
    start=time.perf_counter()
    overrides=dict(model=c["weights"],data=c["data_yaml"],epochs=c["epochs"],
        imgsz=c["size"],batch=c["batch"],device="0,1" if c["gpus"]==2 else "0",
        seed=c["seed"],deterministic=True,workers=c["workers"],amp=c["amp"],
        optimizer="SGD",lr0=c["learning_rate"],lrf=0.01,momentum=0.9,weight_decay=0.0005,
        cos_lr=True,warmup_epochs=c["warmup_epochs"],nbs=c["batch"],
        patience=c["patience"],project=str(Path(c["output"]).parent),
        name=Path(c["output"]).name,exist_ok=True,plots=False,save=True,save_period=10,
        val=True,rect=False,multi_scale=False,mosaic=0.0,mixup=0.0,copy_paste=0.0,
        close_mosaic=0,translate=0.0,scale=0.0,degrees=0.0,shear=0.0,perspective=0.0,
        hsv_h=0.0,hsv_s=0.0,hsv_v=0.0,fliplr=0.0,flipud=0.0,cache=False,
        fraction=1.0,verbose=True,resume=c.get("resume") or False)
    trainer=LabTrainer(overrides=overrides)
    trainer.train()
    if int(os.environ.get("RANK","0"))==0:
        save_json(Path(c["output"])/"training_summary.json",{
            "seconds_this_session":time.perf_counter()-start,"best_mAP50_95":trainer.best_fitness,
            "completed_epochs":trainer.epoch+1,"world_size":c["gpus"],
            "global_batch":c["batch"],"seed":c["seed"],"EMA":True,
            "resume_note":"Native Ultralytics resume; stochastic worker state may differ after resume."})


if __name__=="__main__":
    main()


In [ ]:
# Resume từ output cũ đã attach: giữ best checkpoint và history của run trước.
if RESUME_CHECKPOINT:
    previous_last=Path(RESUME_CHECKPOINT)
    assert previous_last.is_file(), "Không tìm thấy last.pt để resume."
    previous_best=previous_last.parent/"best.pt"
    assert previous_best.is_file(), "Cần giữ cả best.pt và last.pt từ run trước."
    is_yolo=previous_last.parent.name=="weights"
    destination=OUTPUT/"weights/best.pt" if is_yolo else OUTPUT/"best.pt"
    destination.parent.mkdir(parents=True,exist_ok=True)
    if previous_best.resolve()!=destination.resolve():
        shutil.copy2(previous_best,destination)
    old_output=previous_last.parent.parent if is_yolo else previous_last.parent
    for name in ("results.csv","history.csv"):
        source=old_output/name
        if source.exists() and source.resolve()!=(OUTPUT/name).resolve():
            shutil.copy2(source,OUTPUT/name)
# 4. Tải pretrained YOLOv8s một lần trước khi spawn hai process.
from ultralytics import YOLO
download=YOLO("yolov8s.pt")
source=Path("yolov8s.pt").resolve()
if source != (RUNTIME/"yolov8s.pt").resolve():
    shutil.copy2(source,RUNTIME/"yolov8s.pt")
del download

# Chỉ lưu mã và config, tránh copy theo symlink ảnh của GPU smoke test khi resume.
source_output=OUTPUT/"source"
source_output.mkdir(parents=True,exist_ok=True)
for source in RUNTIME.iterdir():
    if source.is_file() and source.suffix in (".py",".json",".yaml"):
        shutil.copy2(source,source_output/source.name)

def run_command(command,env=None):
    print(" ".join(map(str,command)),flush=True)
    process=subprocess.Popen(list(map(str,command)),stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    for line in process.stdout:
        print(line,end="",flush=True)
    status=process.wait()
    if status:
        raise subprocess.CalledProcessError(status,command)


In [ ]:
# 5a. Chạy thử GPU/DDP/AMP và workers trước khi bắt đầu run chính.
sys.path.insert(0,str(RUNTIME))
from common import prepare_gpu_smoke_config
smoke_config=prepare_gpu_smoke_config(CONFIG,RUNTIME)
run_command([sys.executable,"-m","torch.distributed.run","--standalone",
    f"--nproc_per_node={GPUS}",RUNTIME/'yolov8_train.py',"--config",smoke_config])
print("GPU preflight passed. Bắt đầu training đầy đủ ở cell tiếp theo.")


In [ ]:
# 5b. Huấn luyện trên 2 GPU. Nếu OOM: giảm GLOBAL_BATCH và EVAL_BATCH,
# rồi chạy lại từ đầu phần cấu hình; không tự đổi batch ở giữa thí nghiệm.
if not RESUME_CHECKPOINT and any((OUTPUT/name).exists() for name in ("best.pt","last.pt","weights/best.pt","weights/last.pt")):
    raise RuntimeError("Output đã có checkpoint. Chọn OUTPUT mới hoặc đặt RESUME_CHECKPOINT=last.pt.")
run_command([sys.executable,"-m","torch.distributed.run","--standalone",
    f"--nproc_per_node={GPUS}",RUNTIME/'yolov8_train.py',"--config",CONFIG_PATH])


In [ ]:
# 6. Đánh giá best checkpoint trong process riêng, chỉ GPU 0.
# Chạy cell này sau khi training kết thúc; có thể chạy lại mà không train lại.
evaluation_env=os.environ.copy()
evaluation_env["CUDA_VISIBLE_DEVICES"]="0"
run_command([sys.executable,RUNTIME/'yolov8_train.py',"--config",CONFIG_PATH,"--evaluate"],env=evaluation_env)


In [ ]:
# 7. Kết quả chính và hình minh họa (xanh: ground truth; đỏ: prediction).
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image
display(pd.read_csv(OUTPUT/"comparison.csv").T)
display(Image.open(OUTPUT/"test_examples.jpg"))
curve=pd.DataFrame(json.loads((OUTPUT/"validation_PR_F1_curve.json").read_text()))
if not curve.empty:
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    axes[0].plot(curve["recall"],curve["precision"])
    axes[0].set(xlabel="Recall",ylabel="Precision",title="Validation PR (IoU=0.5)",xlim=(0,1),ylim=(0,1))
    axes[1].plot(curve["threshold"],curve["f1"])
    axes[1].set(xlabel="Confidence",ylabel="F1",title="Validation F1")
    fig.tight_layout();fig.savefig(OUTPUT/"validation_curves.png",dpi=150);plt.show()
history_path=OUTPUT/("results.csv" if (OUTPUT/"results.csv").exists() else "history.csv")
history=pd.read_csv(history_path)
history.columns=history.columns.str.strip()
ap_column="val/COCO_mAP50_95" if "val/COCO_mAP50_95" in history else "val_mAP50_95"
if ap_column in history:
    history.plot(x="epoch",y=ap_column,title="Validation COCO mAP@50:95")
    plt.savefig(OUTPUT/"training_AP.png",dpi=150,bbox_inches="tight");plt.show()


In [ ]:
# 8. Gói toàn bộ kết quả để tải xuống từ Kaggle Output.
archive=shutil.make_archive(str(OUTPUT)+"_artifacts","zip",root_dir=OUTPUT)
print("Download:",archive)
print("Để so sánh hai model: ghép comparison.csv từ hai run.")


## Artifact được lưu

- `best.pt` / `weights/best.pt`: checkpoint chọn bằng COCO mAP@50:95 validation.
- `last.pt` / `weights/last.pt`: checkpoint có optimizer để resume trước khi train xong.
- `history.csv` / `results.csv`: loss và validation AP từng epoch.
- `comparison.csv`: đúng 7 metric chính: mAP50:95, AP50, Precision, Recall, F1,
  Mean IoU trên TP và FPS; kèm tên model và seed.
- `metrics.json`: kết quả chi tiết bổ sung để tra cứu khi cần.
- `confidence_selection.json`: ngưỡng chọn trên validation.
- `validation_predictions.json`, `test_predictions.json`: COCO predictions theo tọa độ ảnh gốc.
- `config.json`, `environment.json`, `requirements_freeze.txt`, `source/`: cấu hình và mã của run.
- `test_examples.jpg`, `validation_curves.png`, `training_AP.png`: hình cho báo cáo.

Latency giữa hai run chỉ so sánh khi cùng loại T4, batch 1, FP32 và phạm vi đo.
Không so sánh trực tiếp trị số loss của hai kiến trúc. COCO pretraining và recipe riêng
được công bố; kết quả là so sánh hai hệ thống fine-tune, không chỉ riêng kiến trúc.

Tài liệu: [Torchvision Faster R-CNN](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn.html),
[Ultralytics training](https://docs.ultralytics.com/modes/train/),
[COCO evaluation](https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py).
